In [1]:
!pip install scylla-driver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.6 MB/s  0:00:00m 6.8 MB/s eta 0:00:01


In [3]:
# docker run --name scylla-demo -d -p 9042:9042 scylladb/scylla --smp 2

from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import uuid

def scylla_demo():
    # 1. Connect to the cluster
    cluster = Cluster(['127.0.0.1']) 
    session = cluster.connect()

    print("Connected to ScyllaDB cluster.")

    # 2. Create Keyspace (Database)
    session.execute("""
        CREATE KEYSPACE IF NOT EXISTS demo_keyspace 
        WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
    """)
    session.set_keyspace('demo_keyspace')

    # 3. Create Table
    session.execute("""
        CREATE TABLE IF NOT EXISTS server_metrics (
            server_id uuid,
            timestamp timestamp,
            cpu_temp int,
            fan_speed int,
            PRIMARY KEY (server_id, timestamp)
        ) WITH CLUSTERING ORDER BY (timestamp DESC)
    """)
    print("Table 'server_metrics' is ready.")

    # 4. Insert Data (Prepared Statement for efficiency)
    insert_stmt = session.prepare("""
        INSERT INTO server_metrics (server_id, timestamp, cpu_temp, fan_speed)
        VALUES (?, toTimestamp(now()), ?, ?)
    """)

    my_server_id = uuid.uuid4()
    # Simulating data from SycllaDB
    session.execute(insert_stmt, [my_server_id, 42, 1200])
    session.execute(insert_stmt, [my_server_id, 45, 1250])
    print(f"Inserted metrics for server: {my_server_id}")

    # 5. Query Data
    print("\nRetrieving latest metrics:")
    rows = session.execute("SELECT * FROM server_metrics LIMIT 5")
    for row in rows:
        print(f"Time: {row.timestamp} | Temp: {row.cpu_temp}°C | Fan: {row.fan_speed} RPM")

    cluster.shutdown()

if __name__ == "__main__":
    scylla_demo()

Connected to ScyllaDB cluster.
Table 'server_metrics' is ready.
Inserted metrics for server: 0ab4d257-c271-437a-830f-2467921be439

Retrieving latest metrics:
Time: 2026-01-14 04:36:13.031000 | Temp: 45°C | Fan: 1250 RPM
Time: 2026-01-14 04:36:13.030000 | Temp: 42°C | Fan: 1200 RPM
